# Clean up the results from the new method

This Notebook aims to remove the sentences that are not valid for one reason or another. These reasons include at least the following:
- entity does not refer to the correct target entity, e.g. "HE" as in Higher Education, not as pronoun, or "US" as in United States, not as pronoun, or "palm" as in palm oil, not as Pathways Language Model, among other examples
- the sentence has less than 3 whitespace-delimited tokens -- these are likely to be titles or boilerplate rather than actual sentences that would give any meaningful information
- the sentence is boilerplate that appears in corpus multiple times, e.g. "I did not use generative AI and AI-assisted technologies in the writing process.", appearing 969 times
- the original sentence includes `[MASK]` so it hasn't been scored

In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
dtypes = {
    "year": int,
    "domain": "category",
    "sent_id": int,
    "match_id": int,
    "entity_lower": "category",
    "in_noun_chunk": bool,
    "chunk_root_dep": "category",
    "score": float
}

In [3]:
# If all scores have already been written to a single file, upload from that
new_filename = "/scratch/project_2017974/results-tests/scores/new-method/all-scores_before-cleanup.csv"
df = pd.read_csv(new_filename, dtype=dtypes)

In [50]:
RES_DIR = "/scratch/project_2017974/results-tests/scores/new-method"
score_files = sorted(Path(RES_DIR).glob("*.csv"))
score_files

[PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_001.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_002.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_003.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_004.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_005.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_006.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_007.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_008.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-method/scores_new_method-shard_009.csv'),
 PosixPath('/scratch/project_2017974/results-tests/scores/new-me

In [51]:
df = pd.concat((pd.read_csv(f) for f in score_files), ignore_index=True)
df.info()

/tmp/kapyahos/35586333/ipykernel_2104203/1686595853.py:1: DtypeWarning: Columns (1,3,4,10,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in score_files), ignore_index=True)
/tmp/kapyahos/35586333/ipykernel_2104203/1686595853.py:1: DtypeWarning: Columns (1,3,4,10,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in score_files), ignore_index=True)
/tmp/kapyahos/35586333/ipykernel_2104203/1686595853.py:1: DtypeWarning: Columns (1,3,4,10,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat((pd.read_csv(f) for f in score_files), ignore_index=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3103085 entries, 0 to 3103084
Data columns (total 15 columns):
 #   Column                   Dtype 
---  ------                   ----- 
 0   id                       object
 1   year                     object
 2   domain                   object
 3   sent_id                  object
 4   match_id                 object
 5   sentence                 object
 6   masked_sentence_minimal  object
 7   masked_sentence_chunk    object
 8   entity                   object
 9   entity_lower             object
 10  in_noun_chunk            object
 11  chunk_text               object
 12  chunk_root_text          object
 13  chunk_root_dep           object
 14  score                    object
dtypes: object(15)
memory usage: 355.1+ MB


In [52]:
# Inspect the duplicated rows
# (the assumption is that only redundant column rows should be duplicates)
num_duplicates = len(df[df.duplicated(keep=False)])
df[df.duplicated(keep=False)]

,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score
1897718,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score
1936119,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score
1977657,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score
1993914,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score
2257293,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score
2267406,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score


In [53]:
# Remove the column rows that were accidentally added, shown above
len_before = len(df)
df.drop_duplicates(inplace=True, keep=False, ignore_index=True)
len_after = len(df)

assert (len_before - num_duplicates) == len_after, f"Unexpected length {len_after} after dropping duplicates"

In [54]:
df.to_csv(new_filename, index=False)

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3103079 entries, 0 to 3103078
Data columns (total 15 columns):
 #   Column                   Dtype   
---  ------                   -----   
 0   id                       object  
 1   year                     int64   
 2   domain                   category
 3   sent_id                  int64   
 4   match_id                 int64   
 5   sentence                 object  
 6   masked_sentence_minimal  object  
 7   masked_sentence_chunk    object  
 8   entity                   object  
 9   entity_lower             category
 10  in_noun_chunk            bool    
 11  chunk_text               object  
 12  chunk_root_text          object  
 13  chunk_root_dep           category
 14  score                    float64 
dtypes: bool(1), category(3), float64(1), int64(3), object(7)
memory usage: 272.3+ MB


In [5]:
df

,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,in_noun_chunk,chunk_text,chunk_root_text,chunk_root_dep,score
0,W100457907,2024,Physical Sciences,0,0,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of [MASK...,agent,agent,True,multi-agent systems,systems,pobj,-0.030153
1,W100457907,2024,Physical Sciences,0,1,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of [MASK...,systems,systems,True,multi-agent systems,systems,pobj,-0.104879
2,W100457907,2024,Physical Sciences,0,2,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of multi...,model,model,True,a model driven engineering approach,approach,pobj,-0.051669
3,W100457907,2024,Physical Sciences,0,3,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of multi...,approach,approach,True,a model driven engineering approach,approach,pobj,-0.089126
4,W100457907,2024,Physical Sciences,1,0,Our goal is to cope with the traceability betw...,Our goal is to cope with the traceability betw...,Our goal is to cope with the traceability betw...,models,models,True,implementation models,models,conj,-0.073899
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3103074,W984994666,2008,Social Sciences,10,0,i te ōranga,[MASK] te ōranga,[MASK] te ōranga,i,i,True,i,i,ROOT,0.056573
3103075,W984994666,2008,Social Sciences,11,0,tonutanga e tū tahi ai rāua tahi.,tonutanga e tū tahi [MASK] rāua tahi.,tonutanga e tū tahi [MASK] rāua tahi.,ai,ai,False,NaN,NaN,NaN,0.050651
3103076,W984994666,2008,Social Sciences,15,0,"ake i te tangata whenua, kā rua, me te Taha Ru...","ake [MASK] te tangata whenua, kā rua, me te Ta...","ake [MASK] te tangata whenua, kā rua, me te Ta...",i,i,True,i,i,appos,0.057994
3103077,W984994666,2008,Social Sciences,15,1,"ake i te tangata whenua, kā rua, me te Taha Ru...","ake i te tangata whenua, kā rua, me te Taha Ru...","ake i te tangata whenua, kā rua, me te Taha Ru...",he,he,True,he,he,nsubj,0.062657


In [6]:
# Get the missing HEAD annotations
HEAD_DIR = "/scratch/project_2017974/results-tests/csv-data/masked-sentences/with-head"
head_paths = sorted(Path(HEAD_DIR).glob("*.csv"))
head_df = pd.concat((pd.read_csv(f) for f in head_paths), ignore_index=True)

In [7]:
head_df_head_only = head_df[["id", "sent_id", "match_id", "head"]]

In [8]:
# Get the extended token annotations
TOKEN_DIR = "/scratch/project_2017974/results-tests/csv-data/token-data"
token_paths = sorted(Path(TOKEN_DIR).glob("*.csv"))

In [9]:
token_dtypes = {key: "category" for key in "domain,role,pos,tag,dep,ent_type,ent_iob".split(",")}
token_df = pd.concat((pd.read_csv(f, dtype=token_dtypes) for f in token_paths), ignore_index=True)
token_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11714934 entries, 0 to 11714933
Data columns (total 13 columns):
 #   Column    Dtype   
---  ------    -----   
 0   id        object  
 1   year      int64   
 2   domain    object  
 3   sent_id   int64   
 4   match_id  int64   
 5   role      category
 6   token_i   int64   
 7   text      object  
 8   pos       object  
 9   tag       object  
 10  dep       object  
 11  ent_type  object  
 12  ent_iob   category
dtypes: category(2), int64(4), object(7)
memory usage: 1005.5+ MB


In [10]:
token_df = token_df.astype(token_dtypes)
token_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11714934 entries, 0 to 11714933
Data columns (total 13 columns):
 #   Column    Dtype   
---  ------    -----   
 0   id        object  
 1   year      int64   
 2   domain    category
 3   sent_id   int64   
 4   match_id  int64   
 5   role      category
 6   token_i   int64   
 7   text      object  
 8   pos       category
 9   tag       category
 10  dep       category
 11  ent_type  category
 12  ent_iob   category
dtypes: category(7), int64(4), object(2)
memory usage: 614.5+ MB


In [11]:
match_tokens = token_df[token_df["role"] == "match"].copy().reset_index(drop=True)
match_tokens

,id,year,domain,sent_id,match_id,role,token_i,text,pos,tag,dep,ent_type,ent_iob
0,W100457907,2024,Physical Sciences,0,0,match,9,agent,ADJ,JJ,amod,NaN,O
1,W100457907,2024,Physical Sciences,0,1,match,10,systems,NOUN,NNS,pobj,NaN,O
2,W100457907,2024,Physical Sciences,0,2,match,14,model,NOUN,NN,nmod,NaN,O
3,W100457907,2024,Physical Sciences,0,3,match,17,approach,NOUN,NN,pobj,NaN,O
4,W100457907,2024,Physical Sciences,1,0,match,31,models,NOUN,NNS,conj,NaN,O
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3423528,W984994666,2008,Social Sciences,10,0,match,128,i,PRON,PRP,ROOT,NaN,O
3423529,W984994666,2008,Social Sciences,11,0,match,135,ai,AUX,VBP,aux,NaN,O
3423530,W984994666,2008,Social Sciences,15,0,match,183,i,PRON,PRP,appos,NaN,O
3423531,W984994666,2008,Social Sciences,15,1,match,198,he,PRON,PRP,nsubj,NaN,O


In [12]:
match_tokens_stripped = match_tokens.drop(columns=["year", "domain", "role"])
match_tokens_stripped.rename(columns={"text": "token_text"}, inplace=True)
match_tokens_stripped

,id,sent_id,match_id,token_i,token_text,pos,tag,dep,ent_type,ent_iob
0,W100457907,0,0,9,agent,ADJ,JJ,amod,NaN,O
1,W100457907,0,1,10,systems,NOUN,NNS,pobj,NaN,O
2,W100457907,0,2,14,model,NOUN,NN,nmod,NaN,O
3,W100457907,0,3,17,approach,NOUN,NN,pobj,NaN,O
4,W100457907,1,0,31,models,NOUN,NNS,conj,NaN,O
...,...,...,...,...,...,...,...,...,...,...
3423528,W984994666,10,0,128,i,PRON,PRP,ROOT,NaN,O
3423529,W984994666,11,0,135,ai,AUX,VBP,aux,NaN,O
3423530,W984994666,15,0,183,i,PRON,PRP,appos,NaN,O
3423531,W984994666,15,1,198,he,PRON,PRP,nsubj,NaN,O


In [13]:
merged = df.merge(match_tokens_stripped, on=["id", "sent_id", "match_id"], how="left")
merged

,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,...,chunk_root_text,chunk_root_dep,score,token_i,token_text,pos,tag,dep,ent_type,ent_iob
0,W100457907,2024,Physical Sciences,0,0,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of [MASK...,agent,agent,...,systems,pobj,-0.030153,9,agent,ADJ,JJ,amod,NaN,O
1,W100457907,2024,Physical Sciences,0,1,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of [MASK...,systems,systems,...,systems,pobj,-0.104879,10,systems,NOUN,NNS,pobj,NaN,O
2,W100457907,2024,Physical Sciences,0,2,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of multi...,model,model,...,approach,pobj,-0.051669,14,model,NOUN,NN,nmod,NaN,O
3,W100457907,2024,Physical Sciences,0,3,This paper focuses on the development of multi...,This paper focuses on the development of multi...,This paper focuses on the development of multi...,approach,approach,...,approach,pobj,-0.089126,17,approach,NOUN,NN,pobj,NaN,O
4,W100457907,2024,Physical Sciences,1,0,Our goal is to cope with the traceability betw...,Our goal is to cope with the traceability betw...,Our goal is to cope with the traceability betw...,models,models,...,models,conj,-0.073899,31,models,NOUN,NNS,conj,NaN,O
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3423528,W984994666,2008,Social Sciences,10,0,i te ōranga,[MASK] te ōranga,[MASK] te ōranga,i,i,...,i,ROOT,0.056573,128,i,PRON,PRP,ROOT,NaN,O
3423529,W984994666,2008,Social Sciences,11,0,tonutanga e tū tahi ai rāua tahi.,tonutanga e tū tahi [MASK] rāua tahi.,tonutanga e tū tahi [MASK] rāua tahi.,ai,ai,...,NaN,NaN,0.050651,135,ai,AUX,VBP,aux,NaN,O
3423530,W984994666,2008,Social Sciences,15,0,"ake i te tangata whenua, kā rua, me te Taha Ru...","ake [MASK] te tangata whenua, kā rua, me te Ta...","ake [MASK] te tangata whenua, kā rua, me te Ta...",i,i,...,i,appos,0.057994,183,i,PRON,PRP,appos,NaN,O
3423531,W984994666,2008,Social Sciences,15,1,"ake i te tangata whenua, kā rua, me te Taha Ru...","ake i te tangata whenua, kā rua, me te Taha Ru...","ake i te tangata whenua, kā rua, me te Taha Ru...",he,he,...,he,nsubj,0.062657,198,he,PRON,PRP,nsubj,NaN,O


In [15]:
merged[merged["entity_lower"] == "artificial intelligence"].head(10)

,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,...,chunk_root_text,chunk_root_dep,score,token_i,token_text,pos,tag,dep,ent_type,ent_iob
6,W101656788,2010,Physical Sciences,0,0,Problem statement: Artificial intelligence cha...,Problem statement: [MASK] chatbot is a technol...,Problem statement: [MASK] is a technology that...,Artificial intelligence,artificial intelligence,...,chatbot,nsubj,-0.019933,3,Artificial,ADJ,JJ,amod,NaN,O
7,W101656788,2010,Physical Sciences,0,0,Problem statement: Artificial intelligence cha...,Problem statement: [MASK] chatbot is a technol...,Problem statement: [MASK] is a technology that...,Artificial intelligence,artificial intelligence,...,chatbot,nsubj,-0.019933,4,intelligence,NOUN,NN,compound,NaN,O
23,W1056793151,2015,Physical Sciences,4,0,"However, due to non-deterministic nature of Ar...","However, due to non-deterministic nature of [M...","However, due to non-deterministic nature of [M...",Artificial Intelligence,artificial intelligence,...,Intelligence,pobj,-0.050985,65,Artificial,PROPN,NNP,compound,ORG,B
24,W1056793151,2015,Physical Sciences,4,0,"However, due to non-deterministic nature of Ar...","However, due to non-deterministic nature of [M...","However, due to non-deterministic nature of [M...",Artificial Intelligence,artificial intelligence,...,Intelligence,pobj,-0.050985,66,Intelligence,PROPN,NNP,pobj,ORG,I
63,W1173882893,1985,Social Sciences,0,1,Today's computers come with 'software' program...,Today's computers come with 'software' program...,Today's computers come with 'software' program...,artificial intelligence,artificial intelligence,...,intelligence,nsubj,-0.128128,21,artificial,ADJ,JJ,amod,NaN,O
64,W1173882893,1985,Social Sciences,0,1,Today's computers come with 'software' program...,Today's computers come with 'software' program...,Today's computers come with 'software' program...,artificial intelligence,artificial intelligence,...,intelligence,nsubj,-0.128128,22,intelligence,NOUN,NN,nsubj,NaN,O
66,W1173882893,1985,Social Sciences,2,0,But the ultimate promise of artificial intelli...,But the ultimate promise of [MASK] is far grea...,But the ultimate promise of [MASK] is far grea...,artificial intelligence,artificial intelligence,...,intelligence,pobj,0.017348,68,artificial,ADJ,JJ,amod,NaN,O
67,W1173882893,1985,Social Sciences,2,0,But the ultimate promise of artificial intelli...,But the ultimate promise of [MASK] is far grea...,But the ultimate promise of [MASK] is far grea...,artificial intelligence,artificial intelligence,...,intelligence,pobj,0.017348,69,intelligence,NOUN,NN,pobj,NaN,O
117,W128729319,1995,Physical Sciences,0,0,"Since its inception, artificial intelligence h...","Since its inception, [MASK] has relied upon a ...","Since its inception, [MASK] has relied upon a ...",artificial intelligence,artificial intelligence,...,intelligence,nsubj,-0.039385,4,artificial,ADJ,JJ,amod,NaN,O
118,W128729319,1995,Physical Sciences,0,0,"Since its inception, artificial intelligence h...","Since its inception, [MASK] has relied upon a ...","Since its inception, [MASK] has relied upon a ...",artificial intelligence,artificial intelligence,...,intelligence,nsubj,-0.039385,5,intelligence,NOUN,NN,nsubj,NaN,O


In [16]:
merged.duplicated().sum()

0

In [17]:
# Merge the missing head annotations
merged = merged.merge(head_df_head_only, on=["id", "sent_id", "match_id"], how="left")

In [18]:
merged.entity.nunique()

359

In [19]:
merged.entity_lower.nunique()

71

## Deletion of invalid entities

In [21]:
# Inspect some of the instances of non-uppercase "I", as they are likely
# not referring to "I" as in me
merged[(merged["entity"] != "I") & (merged["entity_lower"] == "i")]

,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,...,chunk_root_dep,score,token_i,token_text,pos,tag,dep,ent_type,ent_iob,head
131,W128729319,1995,Physical Sciences,8,1,"We then construct universal ABO programs, i.e....","We then construct universal ABO programs, [MAS...","We then construct universal ABO programs, [MAS...",i,i,...,appos,0.093732,178,i.e.,X,FW,advmod,NaN,O,program
554,W1500287207,2010,Physical Sciences,4,2,"With an open architecture, the ADN is designed...","With an open architecture, the ADN is designed...","With an open architecture, the ADN is designed...",i,i,...,NaN,0.093738,90,i.e.,X,FW,advmod,NaN,O,NaN
556,W1500287207,2010,Physical Sciences,4,4,"With an open architecture, the ADN is designed...","With an open architecture, the ADN is designed...","With an open architecture, the ADN is designed...",i,i,...,NaN,0.093740,103,i.e.,X,FW,dep,NaN,O,NaN
574,W1500287207,2010,Physical Sciences,32,0,Two fundamental solutions for the minimum cost...,Two fundamental solutions for the minimum cost...,Two fundamental solutions for the minimum cost...,i,i,...,conj,0.093595,762,i.e.,X,FW,advmod,NaN,O,flow
586,W1500287207,2010,Physical Sciences,42,0,The MAS platform is created on different compu...,The MAS platform is created on different compu...,The MAS platform is created on different compu...,i,i,...,dobj,0.093538,996,i.e.,X,FW,advmod,NaN,O,retrieve
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3423525,W984994666,2008,Social Sciences,7,0,"i āna tikanga, ā, kia kaua tētahi e aukati i t...","[MASK] āna tikanga, ā, kia kaua tētahi e aukat...","[MASK] āna tikanga, ā, kia kaua tētahi e aukat...",i,i,...,nsubj,0.037712,95,i,PRON,PRP,nsubj,NaN,O,tētahi
3423526,W984994666,2008,Social Sciences,7,1,"i āna tikanga, ā, kia kaua tētahi e aukati i t...","i āna tikanga, ā, kia kaua tētahi e aukati [MA...","i āna tikanga, ā, kia kaua tētahi e aukati [MA...",i,i,...,appos,0.054855,106,i,PROPN,NNP,appos,NaN,O,i
3423528,W984994666,2008,Social Sciences,10,0,i te ōranga,[MASK] te ōranga,[MASK] te ōranga,i,i,...,ROOT,0.056573,128,i,PRON,PRP,ROOT,NaN,O,i
3423530,W984994666,2008,Social Sciences,15,0,"ake i te tangata whenua, kā rua, me te Taha Ru...","ake [MASK] te tangata whenua, kā rua, me te Ta...","ake [MASK] te tangata whenua, kā rua, me te Ta...",i,i,...,appos,0.057994,183,i,PRON,PRP,appos,NaN,O,ake


In [23]:
merged[(merged["entity"] != "I") & (merged["entity_lower"] == "i")]["chunk_text"].nunique()

9014

In [24]:
merged[(merged["entity"] != "I") & (merged["entity_lower"] == "i")]["token_text"].nunique()

1167

This reveals that regex `\bI\b` produces many, many invalid matches for `I`. With this, though, one could save the instances of `A.I.` and save as said term.

In [27]:
merged[merged["entity"] == "I"]["token_text"].unique()

array(['I', 'A.I.', 'I.', 'I‐494', 'A‐I', 'M.I.T.', '-I', 'C.S.I.C.',
       'I,-75', "I'/'me", 'R.I.S.E.', 'P.I.', 'I&C', 'Intelligence(A.I',
       'A.I', 'I.O.', 'O.I.', 'D.A.I.', 'Z.I.', 'A.I.-based', 'I/', 'I&O',
       'I&O.', 'B.I.', 'y)=1N∑j=1N(gj−⟨g⟩)(Ij−⟨I⟩),(Equation', '⟨I⟩.',
       'non‐I', 'C.I.', 'I.Q.', 'A.L.I.C.E', 'V.I.', 'I‐FP‐CIT', 'I.V.',
       'M.I.', 'A.I.-related', 'A.I.R', 'I).Table', 'I.Y.', 'I.-produced',
       'D.I.', '""I', 'I^3', 'I‐CANe', 'Australia?I', 'I.L.',
       'EverythiA.I.ng', 'D&I', 'I**mages', 'I.A.', 'SADEA‐I', 'SADEA‐I.',
       "I've", 'I.S.', 'R.I.', 'I-131', 'sentient?—I', 'I.T',
       'A.L.I.C.E.', '"I', '5)I', 'I.o', 'A.I.)-based', 'A.I.-driven',
       'https://github.com/jiaxiaojunQAQ/I-GCG',
       'https://github.com/sail-sg/I-FSJ', 'I-', 'I&M', 'C.I', 'I:-',
       'heteronormativity).I', "awesome!I'm", 'start?I', "I''~-~the",
       'M.A.I.', 'sA.I.', 'R&D&I', "SecureSQL.I'm", 'better?I', "I'jaaz",
       'I.e.', 'G="I', 'I.P.',

In [30]:
merged[(merged["entity"] == "I") & (merged["token_text"] == "I") & (merged["pos"] != "PRON")]

,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,...,chunk_root_dep,score,token_i,token_text,pos,tag,dep,ent_type,ent_iob,head
1105,W1533904683,1994,Life Sciences,0,1,Liver-specific expression of the apolipoprotei...,Liver-specific expression of the apolipoprotei...,Liver-specific expression of the apolipoprotei...,I,i,...,appos,0.077338,11,I,NOUN,NN,appos,NaN,O,ai
1106,W1533904683,1994,Life Sciences,0,2,Liver-specific expression of the apolipoprotei...,Liver-specific expression of the apolipoprotei...,Liver-specific expression of the apolipoprotei...,I,i,...,pobj,0.053467,55,I,VERB,VBN,compound,NaN,O,of
1107,W1533904683,1994,Life Sciences,2,0,"In the current report, enhancer derivatives wi...","In the current report, enhancer derivatives wi...","In the current report, enhancer derivatives wi...",I,i,...,dobj,0.007970,107,I,NUM,CD,prt,NaN,O,stimulate
1108,W1533904683,1994,Life Sciences,5,0,Binding of HNF-3 beta to site B transactivates...,Binding of HNF-3 beta to site B transactivates...,Binding of HNF-3 beta to site B transactivates...,I,i,...,appos,0.052718,170,I,NUM,CD,prt,NaN,O,bind
1110,W1533904683,1994,Life Sciences,7,1,"B. Furthermore, HNF-3 beta and HNF-4, bound to...","B. Furthermore, HNF-3 beta and HNF-4, bound to...","B. Furthermore, HNF-3 beta and HNF-4, bound to...",I,i,...,pobj,0.065233,226,I,NUM,CD,compound,NaN,O,of
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3421105,W7153633322,2026,Physical Sciences,39,0,"All component theorems (Parts I–V, Theorems 28...","All component theorems (Parts [MASK]–V, Theore...","All component theorems ([MASK]–V, Theorems 28....",I,i,...,nsubj,0.046569,858,I,NOUN,NN,nsubj,NaN,O,be
3421216,W7153719202,2026,Health Sciences,27,0,The KLASS-01 trial was pivotal in establishing...,The KLASS-01 trial was pivotal in establishing...,The KLASS-01 trial was pivotal in establishing...,I,i,...,conj,0.019572,757,I,PROPN,NNP,compound,NaN,O,treatment
3421617,W7153801219,2026,Physical Sciences,9,0,(Phase 60) v1–v6 Foundations (Phases 1–38) (I)...,(Phase 60) v1–v6 Foundations (Phases 1–38) ([M...,(Phase 60) v1–v6 Foundations (Phases 1–38) [MA...,I,i,...,appos,0.042320,357,I,NOUN,NN,nmod,NaN,O,foundations
3423398,W862450444,2005,Physical Sciences,8,1,"Also, we then examined of the Metal-Insulator-...","Also, we then examined of the Metal-Insulator-...","Also, we then examined of the Metal-Insulator-...",I,i,...,pobj,0.086860,179,I,PROPN,NNP,pobj,NaN,O,of


In [2]:
def print_examples(df: pd.DataFrame, n=20):
    for row in df.sample(n, random_state=999).itertuples():
        print(row.sentence)
        print(row.masked_sentence_minimal)
        print()

In [33]:
# Print examples of invalid "I"s
print_examples(merged[(merged["entity"] == "I") & (merged["token_text"] == "I") & (merged["pos"] != "PRON")])

Parallel assessment of pepsinogen I (PG I), pepsinogen II (PG II), PG I/PG II ratio and gastrin-17 (G-17), as well as antibodies to Helicobacter pylori is an exact and validated set of stomach-specific biomarkers that reflect the extent and grade of gastric atrophy as a main premalignant condition for GC.
Parallel assessment of pepsinogen I (PG [MASK]), pepsinogen II (PG II), PG I/PG II ratio and gastrin-17 (G-17), as well as antibodies to Helicobacter pylori is an exact and validated set of stomach-specific biomarkers that reflect the extent and grade of gastric atrophy as a main premalignant condition for GC.

Digital cognitive behavioral therapy for insomnia (dCBT-I) is effective in treating insomnia, but adherence remains a major challenge in real-world applications.
Digital cognitive behavioral therapy for insomnia (dCBT-[MASK]) is effective in treating insomnia, but adherence remains a major challenge in real-world applications.

In this paper, we explain that since the Civil Rig

In [35]:
# Print examples of possibly valid "I"s
print_examples(merged[(merged["entity"] == "I") & (merged["token_text"] == "I") & (merged["pos"] == "PRON")])

By analyzing current barriers and emerging trends, I propose a practical roadmap for integration AI into employee tools in an ethical, inclusive and outcome -oriented way.
By analyzing current barriers and emerging trends, [MASK] propose a practical roadmap for integration AI into employee tools in an ethical, inclusive and outcome -oriented way.

First, I point out AI tools' potential benefits in academia.
First, [MASK] point out AI tools' potential benefits in academia.

In the third chapter, I argue that a syntactic parser, in particular my approach which draws a clear division between competence and performance, can be used as a model to explore formulaic and creative language use, starting with Sinclair's (1991) distinction between idiom principle and syntax principle, and ending with the suggestion to use the parser as a psycholinguistic model.
In the third chapter, [MASK] argue that a syntactic parser, in particular my approach which draws a clear division between competence and

In [36]:
# Inspect the cases when I is annotated as proper noun instead of a pronoun
print_examples(merged[(merged["entity"] == "I") & (merged["token_text"] == "I") & (merged["pos"] == "PROPN")])

While a conservation-like balance at the global scale (Δ(E + I + A)
While a conservation-like balance at the global scale (Δ(E + [MASK] + A)

Selected buffaloes were grouped, Group I and II (n=40 each) buffaloes were treated with CIDR intravaginal application for 7 and 9 days respectively followed by same treatment of Inj.
Selected buffaloes were grouped, Group [MASK] and II (n=40 each) buffaloes were treated with CIDR intravaginal application for 7 and 9 days respectively followed by same treatment of Inj.

This paper introduces the Integrated Analysis Matrix (I-AM), a novel hybrid AI-assisted and human-led methodological approach towards the mindful integration, analysis, and interpretation of complex multimodal, mixed methods datasets – applied here to educational research, particularly where Game-based Learning (GBL) environments generate large volumes of entangled qualitative and quantitative data.
This paper introduces the Integrated Analysis Matrix ([MASK]-AM), a novel hybrid AI

In [38]:
# Inspect the "A.I."s from the entity "I"
print_examples(merged[merged["token_text"] == "A.I."])

This research focuses on exploring the role of A.I. in improving the quality of foreign language education, thereby proposing specific A.I. applications.
This research focuses on exploring the role of A.I. in improving the quality of foreign language education, thereby proposing specific A.[MASK]. applications.

The website (corrosiondetector.com) permits any user to label images, with such labelling then contributing to the training of a cloud based A.I. model - with such a cloud-based model then capable of assessing any fresh (or uploaded) image for the presence of corrosion.
The website (corrosiondetector.com) permits any user to label images, with such labelling then contributing to the training of a cloud based A.[MASK]. model - with such a cloud-based model then capable of assessing any fresh (or uploaded) image for the presence of corrosion.

Additionally, our article discusses the application of A.I. in private prisons versus state prisons, thereby revealing its ethical implica

In [39]:
(merged["token_text"] == "A.I.").sum()

414

Conclusion: A.I.s cannot be recovered as the score has been calculated only for the `I` part. But as there are only 414 of them, that is not a massive loss.

In [59]:
# Remove the "I" instances that are not a pronoun and include some other text in the annotated token
to_remove_i = ((merged["entity_lower"] == "i") & ((merged["token_text"] != "I") | (merged["pos"] != "PRON")))
merged = merged[~to_remove_i]

In [61]:
merged[(merged["entity_lower"] == "i")]["token_text"].unique()

array(['I'], dtype=object)

In [63]:
merged[(merged["entity_lower"] == "i")]["entity"].unique()

array(['I'], dtype=object)

### Remove human entities that are not annotated as pronouns

In [4]:
def get_entities(filename:str) -> list[str]:
    with open(filename) as file:
        return set((line.strip() for line in file if line.strip()))

TARGET_ENTS_DIR = "/scratch/project_2017974/results-tests/target_entities/"
HUMAN = get_entities(TARGET_ENTS_DIR + "entities_human.txt")
LM = get_entities(TARGET_ENTS_DIR + "entities_lm.txt")
ARTIFACT = get_entities(TARGET_ENTS_DIR + "entities_artifact.txt")

In [5]:
for ent_list, name in zip((HUMAN, LM, ARTIFACT), ("Human", "LM", "Artifact")):
    print(f"{name} terms ({len(ent_list)}): {sorted(ent_list)}")
    print()

Human terms (20): ['author', 'authors', 'companies', 'he', 'her', 'him', 'humans', 'i', 'organizations', 'participants', 'patients', 'people', 'researchers', 'she', 'students', 'teachers', 'us', 'user', 'users', 'we']

LM terms (25): ['bert', 'chatbot', 'chatbots', 'chatgpt', 'claude', 'foundation model', 'foundation models', 'gemini', 'genai', 'gpt', 'language model', 'language models', 'large language model', 'large language models', 'llama', 'llm', 'llms', 'lm', 'lms', 'palm', 'pre-trained language model', 'pre-trained language models', 'transformer', 'transformers', 'xlnet']

Artifact terms (21): ['agent', 'agents', 'ai', 'algorithm', 'algorithms', 'application', 'applications', 'approach', 'approaches', 'architecture', 'artificial intelligence', 'framework', 'frameworks', 'model', 'models', 'network', 'networks', 'software', 'system', 'systems', 'technology']



In [66]:
def assign_entity_type(entity: str):
    if entity in HUMAN:
        return "human"
    elif entity in LM:
        return "lm"
    elif entity in ARTIFACT:
        return "artifact"
    else:
        return pd.NA

In [67]:
merged["entity_type"] = merged["entity_lower"].apply(assign_entity_type)

In [68]:
pronouns = set("he her him i she us we".split())

In [71]:
# Remove pronouns that are not annotated as pronouns
merged = merged[~((merged["entity_lower"].isin(pronouns)) & (merged["pos"] != "PRON"))]

### Remove sentences that do not have scores

(This could have been the first thing to do, but better late than never)

In [74]:
merged = merged.dropna(subset="score").copy()

## AI removal

Maybe the trickiest part is to remove the invalid instances of AI.

In [76]:
# Inspect the instances of AI that are not written in uppercase
print_examples(merged[(merged["entity"] != "AI") & (merged["entity_lower"] == "ai")])

riscontro nel riconoscimento professionale e dovendo rispondere a precise norme ministeriali in merito alla costruzione dei curricula formativi offerti ai vari livelli, dai diplomi di laurea ai corsi di dottorato, non possono che basare le proprie aperture didattiche all'educazione
riscontro nel riconoscimento professionale e dovendo rispondere a precise norme ministeriali in merito alla costruzione dei curricula formativi offerti [MASK] vari livelli, dai diplomi di laurea ai corsi di dottorato, non possono che basare le proprie aperture didattiche all'educazione

However, root collar diameter (RCD) increased for hexazinone applied at 0.56 and 1.12 kg ai/ha, and sulfometuron methyl applied at 0.42 kg ai/ha at the end of the first growing season compared to the control.
However, root collar diameter (RCD) increased for hexazinone applied at 0.56 and 1.12 kg [MASK]/ha, and sulfometuron methyl applied at 0.42 kg ai/ha at the end of the first growing season compared to the control.

This r

In [78]:
# Remove said instances, as they do not seem to be about "AI"
# as in "artificial intelligence"
merged = merged[~((merged["entity"] != "AI") & (merged["entity_lower"] == "ai"))]

In [86]:
print_examples(merged[(merged["entity_lower"] == "ai") & (merged["token_text"] != "AI")])

Total body IR resulted in histological testicular damage (decrease in Johnsen's criteria, p<0.05) that was accompanied by a significant increase in germ cell apoptosis, expressed as the number of apoptotic cells per 100 tubules (AI-1 apoptotic index) and the number of positive tubules per 100 tubules (AI-2 apoptotic index).
Total body IR resulted in histological testicular damage (decrease in Johnsen's criteria, p<0.05) that was accompanied by a significant increase in germ cell apoptosis, expressed as the number of apoptotic cells per 100 tubules (AI-1 apoptotic index) and the number of positive tubules per 100 tubules ([MASK]-2 apoptotic index).

AcknowledgmentWe acknowledge support of this work by the project "Research Infrastructure on Food Bioprocessing Development and Innovation Exploitation – Food Innovation RI" (MIS 5027222), which is implemented under the Action "Reinforcement of the Research and Innovation Infrastructure", funded by the Operational Programme "Competitiveness,

In [87]:
print_examples(merged[(merged["entity_lower"] == "ai") & (merged["token_text"] == "AI")])

NLP is an area of AI dealing with the ability of a computer program to understand human language as it is spoken and written—referred to as natural language.
NLP is an area of [MASK] dealing with the ability of a computer program to understand human language as it is spoken and written—referred to as natural language.

The integration of AI into patient-centered care is examined, focusing on personalized medicine, clinical decision support, and AI-powered patient engagement tools that improve counseling and patient monitoring.
The integration of [MASK] into patient-centered care is examined, focusing on personalized medicine, clinical decision support, and AI-powered patient engagement tools that improve counseling and patient monitoring.

The benefits and challenges of AI adoption in disaster management are also highlighted.
The benefits and challenges of [MASK] adoption in disaster management are also highlighted.

This framework grew out of lived experience and has been refined thro

In [92]:
merged[(merged["entity_lower"] == "ai")]["token_text"].value_counts().head(50)

token_text
AI                                                  516721
AI)-based                                             1394
AI-2                                                   888
AI)-driven                                             665
AI)/acre                                               593
AI-                                                    460
AI‐based                                               310
AI)-assisted                                           264
AI‐driven                                              261
AI)-powered                                            244
AI‐generated                                           185
AI)-enabled                                            176
AI‐assisted                                            128
AI‐powered                                             101
AI]/acre                                               100
intelligence(AI                                         92
AI‐enabled                                   

In [97]:
# Hand-pick tokens from the most common ones that seem most likely to be referring to an actual AI entity
ai_tokens_to_include = set(
    "AI AI)-based AI)-driven AI- AI‐based AI)-assisted AI‐driven AI)-powered AI‐generated AI‐assisted AI‐powered intelligence(AI AI‐enabled AI)-generated".split()
)
ai_tokens_to_include

{'AI',
 'AI)-assisted',
 'AI)-based',
 'AI)-driven',
 'AI)-generated',
 'AI)-powered',
 'AI-',
 'AI‐assisted',
 'AI‐based',
 'AI‐driven',
 'AI‐enabled',
 'AI‐generated',
 'AI‐powered',
 'intelligence(AI'}

In [98]:
print_examples(merged[merged["token_text"].isin(ai_tokens_to_include)])

Both FDKH dosages significantly improved AI compared to the placebo, with FDKH 250 mg/kg showing the lowest AI value.
Both FDKH dosages significantly improved [MASK] compared to the placebo, with FDKH 250 mg/kg showing the lowest AI value.

Particular attention is given to the myth of AI objectivity and to the question of how digital myth-making processes differ from earlier forms of ideological reality construction.
Particular attention is given to the myth of [MASK] objectivity and to the question of how digital myth-making processes differ from earlier forms of ideological reality construction.

This comparative analysis underlines the need for ongoing global dialogue to harmonize AI regulations that safeguard human rights while promoting technological advancement, reflecting the diverse perspectives and priorities of each region.
This comparative analysis underlines the need for ongoing global dialogue to harmonize [MASK] regulations that safeguard human rights while promoting tech

In [103]:
# Remove instances of AI of which tokens are not within the hand-picked sample
merged = merged[~((merged["entity_lower"] == "ai") & ~(merged["token_text"].isin(ai_tokens_to_include)))].copy()

### Remove short sentences

In [104]:
merged["num_tokens"] = merged["sentence"].str.count(" ") + 1


In [105]:
merged["num_tokens"].describe(percentiles=[0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99])

count    3.375749e+06
mean     2.763416e+01
std      1.857029e+01
min      1.000000e+00
1%       9.000000e+00
5%       1.300000e+01
10%      1.500000e+01
25%      1.900000e+01
50%      2.500000e+01
75%      3.200000e+01
90%      4.100000e+01
95%      4.900000e+01
99%      7.800000e+01
max      2.020000e+03
Name: num_tokens, dtype: float64

In [110]:
merged[merged["num_tokens"] < 3]["sentence"].value_counts().head(30)

sentence
Design/Methodology/Approach:    62
Approach:                       44
Approach.                       38
Theoretical Framework:          34
AI:                             26
The Authors.                    24
Artificial Intelligence         18
Artificial Intelligence:        16
AI.                             12
Artificial Intelligence.        12
AI                              10
Artificial intelligence         10
Author's Note                    9
Design/methodology/approach.     8
I                                7
Design/Methodology/Approach.     6
Artificial intelligence.         6
The Author(s).                   6
AI application.                  6
Applications:                    6
Design/methodology/approach      5
Theoretical Framework            4
Methodology/Approach:            4
Vision-Language Models(VLMs)     4
Artificial intelligence».        4
GPT-3.5                          4
Participants                     3
(AI).                            3
algorithm. 

In [111]:
# Remove sentences that contain less than three whitespace-delimited tokens
merged = merged[merged["num_tokens"] > 2].copy()

In [112]:
merged.entity_type.value_counts()

entity_type
artifact    2200753
human        693483
lm           480648
Name: count, dtype: int64

In [113]:
merged.groupby("entity_type", observed=True)["score"].mean()

entity_type
artifact   -0.041684
human       0.069911
lm         -0.026100
Name: score, dtype: float64

In [114]:
merged[merged["entity_type"] == "lm"].groupby("domain", observed=True)["score"].mean()

domain
Health Sciences     -0.012911
Life Sciences       -0.030283
Physical Sciences   -0.029784
Social Sciences     -0.019877
Name: score, dtype: float64

In [107]:
merged[merged["entity_type"] == "lm"].groupby(["domain", "year"], observed=True)["score"].mean()

domain           year
Health Sciences  1992    0.005078
                 2004    0.012429
                 2008    0.080006
                 2011   -0.003810
                 2012    0.052831
                           ...   
Social Sciences  2022   -0.023460
                 2023   -0.019458
                 2024   -0.019986
                 2025   -0.019824
                 2026   -0.019076
Name: score, Length: 119, dtype: float64

In [115]:
merged.to_csv("new-method-cleanup-wip.csv", index=False)

In [116]:
merged.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3374908 entries, 0 to 3423531
Data columns (total 25 columns):
 #   Column                   Dtype   
---  ------                   -----   
 0   id                       object  
 1   year                     int64   
 2   domain                   category
 3   sent_id                  int64   
 4   match_id                 int64   
 5   sentence                 object  
 6   masked_sentence_minimal  object  
 7   masked_sentence_chunk    object  
 8   entity                   object  
 9   entity_lower             category
 10  in_noun_chunk            bool    
 11  chunk_text               object  
 12  chunk_root_text          object  
 13  chunk_root_dep           category
 14  score                    float64 
 15  token_i                  int64   
 16  token_text               object  
 17  pos                      category
 18  tag                      category
 19  dep                      category
 20  ent_type                 cate

## Continue

In [5]:
# Continue cleanup from the above csv
cat_cols = "domain,entity_lower,chunk_root_dep,pos,tag,dep,ent_type,ent_iob,entity_type".split(",")
dtypes = {col: "category" for col in cat_cols}
df = pd.read_csv("new-method-cleanup-wip.csv", dtype=dtypes)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3374908 entries, 0 to 3374907
Data columns (total 25 columns):
 #   Column                   Dtype   
---  ------                   -----   
 0   id                       object  
 1   year                     int64   
 2   domain                   category
 3   sent_id                  int64   
 4   match_id                 int64   
 5   sentence                 object  
 6   masked_sentence_minimal  object  
 7   masked_sentence_chunk    object  
 8   entity                   object  
 9   entity_lower             category
 10  in_noun_chunk            bool    
 11  chunk_text               object  
 12  chunk_root_text          object  
 13  chunk_root_dep           category
 14  score                    float64 
 15  token_i                  int64   
 16  token_text               object  
 17  pos                      category
 18  tag                      category
 19  dep                      category
 20  ent_type                

In [6]:
# Inspect the different spellings to hand-pick the suspicious ones
with pd.option_context("display.max_rows", None):
    display(df.entity.value_counts())

entity
AI                             520880
artificial intelligence        257150
we                             207352
model                          183505
models                         151113
We                             136845
systems                        131782
system                         115985
approach                        88513
framework                       84720
LLMs                            80876
technology                      76526
Artificial Intelligence         72504
large language models           67371
applications                    58694
students                        50335
patients                        48096
application                     46884
approaches                      45568
network                         44576
Large Language Models           42600
algorithms                      41916
LLM                             40640
user                            39826
language models                 34040
algorithm                       33301
netwo

In [8]:
with pd.option_context("display.max_rows", None):
    display(df.entity_lower.value_counts())

entity_lower
ai                             520880
artificial intelligence        355484
we                             344219
model                          191373
models                         153475
systems                        137464
large language models          123444
system                         123050
approach                        89133
framework                       87871
technology                      83001
llms                            80952
applications                    60253
network                         52347
students                        52304
patients                        50368
application                     48206
approaches                      45834
algorithms                      42656
user                            41164
llm                             40790
language models                 39704
networks                        39336
algorithm                       34909
agent                           34217
chatgpt                         31617

In [28]:
valid_uppercase = set(("AI", "BERT", "I", "GPT", "LLM", "LM"))

Uppercased entities (that are not typically written in all caps) have a high potential to be abbreviations of something else than what they are intended to refer to, or titles.

In [15]:
upper_entities = [term for term in df["entity"].unique() if term.isupper() and term not in valid_uppercase]
print(sorted(upper_entities))

['AGENT', 'AGENTS', 'ALGORITHM', 'ALGORITHMS', 'APPLICATION', 'APPLICATIONS', 'APPROACH', 'APPROACHES', 'ARCHITECTURE', 'ARTIFICIAL INTELLIGENCE', 'AUTHOR', 'AUTHORS', 'Aİ', 'CHATBOT', 'CHATBOTS', 'CHATGPT', 'CLAUDE', 'COMPANIES', 'FRAMEWORK', 'GEMINI', 'GENAI', 'HE', 'HER', 'HUMANS', 'LLAMA', 'LLMS', 'LMS', 'MODEL', 'MODELS', 'NETWORK', 'NETWORKS', 'PALM', 'PARTICIPANTS', 'PATIENTS', 'PEOPLE', 'RESEARCHERS', 'SHE', 'SOFTWARE', 'STUDENTS', 'SYSTEM', 'SYSTEMS', 'TEACHERS', 'TECHNOLOGY', 'TRANSFORMER', 'USER', 'USERS', 'WE', 'XLNET', 'İ']


In [10]:
len(df[df["entity"].isin(upper_entities)])

1480

In [11]:
print_examples(df[df["entity"].isin(upper_entities)])

PREDICTING THE SHEAR CAPACITY OF REINFORCED CONCRETE SLENDER BEAMS WITHOUT STIRRUPS BY APPLYING ARTIFICIAL INTELLIGENCE ALGORITHMS - 8th International Conference on Computational Methods in Structural Dynamics and Earthquake Engineering - 28-30 June 2021, Streamed from Athens, Greece
PREDICTING THE SHEAR CAPACITY OF REINFORCED CONCRETE SLENDER BEAMS WITHOUT STIRRUPS BY APPLYING [MASK] ALGORITHMS - 8th International Conference on Computational Methods in Structural Dynamics and Earthquake Engineering - 28-30 June 2021, Streamed from Athens, Greece

Importantly, PALM provides an interactive frontend that visualizes path coverage alongside generated tests, assembling tests based on the specific paths they exercise.
Importantly, [MASK] provides an interactive frontend that visualizes path coverage alongside generated tests, assembling tests based on the specific paths they exercise.

It examines how LMS platforms support effective communication between educators and students, foster collab

In [18]:
# It seems that GEMINI (written in uppercase) might be always referring to Google's Gemini,
# except for in sentences 24-34 where it refers to a dataset
for i, sent in enumerate(df[df["entity"] == "GEMINI"]["sentence"], start= 1):
    print(f"{i:>2}) {sent}")

 1) [1].Our experiments involve evaluating several proprietary and open-source LLMs (GPT4, GEMINI PRO, OPENCHAT, LLAMA) for stereotypical bias and examining the attributes that influence bias.
 2) The authors tested the COPILOT and GEMINI chatbots.
 3) In this paper, we propose to utilize vision language models (VLMs) such as generative pre-trained transformer (GPT), GEMINI, large language and vision assistant (LLAVA), PaliGemma, and Microsoft Florence2 to recognize facial attributes such as race, gender, age, and emotion from images with human faces.
 4) The most notable improvements between Naive RAG and Advanced RAG are observed in the GEMINI 1.5 and COMMAND R+ variants.
 5) The findings ascertained that AI tools such as ChatGPT, Quizbot, Quizlet, ExamSoft, GEMINI, and Gemini AI were commonly used for exam preparation.
 6) Using Soft prompting and Chain-of-Thought (CoT) prompting, we evaluated the effectiveness of GPT-4, GEMINI, and ClaudeAI in detecting six types of visual defects.

In [19]:
for i, sent in enumerate(df[df["entity"] == "CLAUDE"]["sentence"], start=1):
    print(f"{i:>2}) {sent}")

 1) Among the models, the best-performing variants are CLAUDE 3 OPUS and GPT-4O. These models consistently achieve the highest scores, indicating robust integration and utilization of contextual information.
 2) This paper introduces MultiLLM-Chatbot, a scalable RAG-based benchmarking framework designed to evaluate five popular LLMs GPT-4-Turbo, CLAUDE-3.7-Sonnet, LLAMA-3.3-70B, DeepSeek-R1-Zero, and Gemini-2.0-Flash across five domains: Agriculture, Biology, Economics, Internet of Things (IoT), and Medical.
 3) Description Context-Driven AI Orchestration Engineering for Rapid Full-Stack Delivery documents two greenfield case studies demonstrating how comprehensive project-level context provision (3,500+ line CLAUDE.md documents) and multi-agent AI orchestration (3-role system: Product Manager, App Developer, Backend Developer) enabled rapid delivery of production-grade mobile and backend applications.
 4) No control groups or A/B comparisons Baseline estimates derived from industry pr

In [21]:
for i, sent in enumerate(df[df["entity"] == "CHATGPT"]["sentence"], start=1):
    print(f"{i:>2}) {sent}")

 1) THE ESSENCE OF AI AND CHATGPT In essence, AI is the use of computer science in solving real-world problems, and includes robotics, speech recognition, image recognition, natural language processing, and expert systems[5].
 2) ARTIFICIAL INTELLIGENCE AND CHEMISTRY TEACHING: A PROPAEDEUTIC ANALYSIS OF CHATGPT IN CHEMICAL CONCEPTS DEFINING.
 3) CHATGPT is a well-known conversational artificial intelligence (AI) system based on the generative pre-trained transformer (GPT) architecture, launched by OpenAI.
 4) CHATGPT is trained through reinforcement learning based on human feedback.
 5) There are advantages to the use of CHATGPT in Civil engineering, including but not limited to design and planning: structural analysis and simulation, code compliance and regulations construction management, knowledge repository and information retrieval, education, and research.
 6) The limitation of CHATGPT is the bias based on the datasets used in CHATGPT training, the requirement of sufficient input

In [23]:
for i, sent in enumerate(df[df["entity"] == "LLAMA"]["sentence"], start=1):
    if i > 20:
        break
    print(f"{i:>2}) {sent}")

 1) To address this challenge, we introduce TrafficSafetyGPT, a novel LLAMA-based model, which has undergone supervised fine-tuning using TrafficSafety-2K dataset which has human labels from government produced guiding books and ChatGPT-generated instruction-output pairs.
 2) Our evaluation, centered on LLMs such as GPT-3.5/4, PaLM-2, and LLAMA-2, incorporates advanced prompting strategies like the chain-of-thought (CoT) and tree-of-thought (ToT).
 3) Our study reveals that the effectiveness of the advanced prompting strategies can be inconsistent, occasionally damaging LLM performance, especially in smaller models like the LLAMA-2 (13b).
 4) We evaluate our approach for iT2I in a variety of common-used scenarios under different LLMs, e.g., ChatGPT, LLAMA, Baichuan, and InternLM.
 5) On open-source models such as VICUNA-7B and LLAMA-2-CHAT 7B AND 13B, it shows an attack success rate of more than 91%.
 6) On bias evaluations, Unalignment exposes inherent biases in safety-aligned models 

In [29]:
valid_uppercase.update(("GEMINI", "CLAUDE", "CHATGPT", "LLAMA"))

In [30]:
valid_uppercase

{'AI', 'BERT', 'CHATGPT', 'CLAUDE', 'GEMINI', 'GPT', 'I', 'LLAMA', 'LLM', 'LM'}

In [36]:
for i, sent in enumerate(df[df["entity"] == "LLMS"].sample(30)["sentence"], start=1):
    print(f"{i:>2}) {sent}")

 1) To address the critical need for a large number of diverse visualization items in education and research, this paper investigates the potential for large language models (LLMS) to automate the generation of multiple-choice visualization items.
 2) This review article highlights the paradigm shift brought in by LLMS in NLP, coupling their capabilities and applications with related challenges.
 3) The main contributions of this study are the LLM and LLMS cybersecurity models, the formalization of the results of these experiments, an assessment of the criticality level for cyber risks of the models, and the choice of countermeasures based on the coefficient of their effectiveness.
 4) Large language models(LLMS)have shown excellent text generation capabilities, capable of generating fluent human-like responses for many downstream tasks.
 5) The research results: the basic high-level architecture of LLMS, which consists of external sources, the LLM service, server functions, and storag

In [37]:
for i, sent in enumerate(df[df["entity"] == "LMS"].sample(30)["sentence"], start=1):
    print(f"{i:>2}) {sent}")

 1) Learning Management Systems (LMS), virtual labs, interactive simulations, and AI-powered tutoring platforms not only facilitate access to content but also create dynamic, personalized learning experiences.
 2) [1].The lack of error messages in Canvas LMS exacerbates the issue, making this guidance crucial for educators.
 3) For example, it focuses on the unilateral view of the evaluation method; it doesn't show weights for each different criterion effecting LMS and doesn't rank its importance.
 4) A study was conducted in Hyderabad, encompassing three phases: initial data collection on demographics and learning preferences, adaptation of content to these preferences within a personalized learning management system (LMS), and subsequent evaluation through a survey completed by approximately 67 Defence SME employees.
 5) Digital transformation in education management encompasses administrative processes, learning systems, and student services, leveraging technologies like Learning Ma

In [39]:
# LLMS seems to often refer to Large Language Models,
# whereas LMS are more likely to refer to Learning Management Systems
# than language models, so only keep LLMS
valid_uppercase.add("LLMS")

In [41]:
upper_entities = [ent for ent in upper_entities if ent not in valid_uppercase]

In [42]:
df[df["entity"].isin(upper_entities)]["entity"].value_counts()

entity
LMS                        515
ARTIFICIAL INTELLIGENCE    158
HE                         130
SYSTEM                      41
PALM                        39
MODEL                       36
TECHNOLOGY                  30
AGENT                       27
FRAMEWORK                   26
PATIENTS                    22
WE                          22
SYSTEMS                     20
APPLICATIONS                17
AUTHOR                      17
HER                         15
XLNET                       12
GENAI                       10
MODELS                       9
PARTICIPANTS                 8
AUTHORS                      8
ARCHITECTURE                 7
APPLICATION                  7
USER                         6
CHATBOTS                     6
İ                            6
APPROACH                     6
Aİ                           6
CHATBOT                      5
ALGORITHMS                   5
AGENTS                       5
STUDENTS                     5
RESEARCHERS                  4
N

In [44]:
len(df[df["entity"].isin(upper_entities)])

1262

In [43]:
for i, sent in enumerate(df[df["entity"].isin(upper_entities)].sample(50)["sentence"], start=1):
    print(f"{i:>2}) {sent}")

 1) The rise of digital technologies in every job has reinforced the importance of eLearning as one of the ongoing big players in the domain of corporate training solutions, leading organisations to make the transition from face-to-face training modules and are instead rely on these trusty virtual training tools to impart scalable, affordable, and accessible learning experience; this transition has proven valuable considering the fact that remote work and hybrid work models are here to stay, and according to a recent 2023 report by the Association for Talent Development (ATD), over 70% of organisations today use at least some form of digital learning to upskill employees, which also means that it becomes imperative on the part of companies to effectively implement their eLearning platform and thoroughly engage their employees with it, so that the training content is retained with the employees, and for that, it is important that the content delivery methods are designed based on trends

As there aren't that many all-caps entities, remove all of them (excluding the explicitly wanted entities in all caps).

In [45]:
df = df[~(df["entity"].isin(upper_entities))].copy()

Also, get rid of entities (with any casing) that appear only a few times in the full corpus. This will remove instances of `PaLM` and `XLNet`, which were used in AnthroScore, but they do not seem to be well-represented in this corpus. `him`, despite having only some mentions, is kept as a counterpart for `her` (which likely includes equivalents to both `his` and `him`).

In [50]:
df.entity_lower.value_counts()[lambda x: x < 1000]

entity_lower
him                        686
palm                       652
xlnet                      174
artificial ıntelligence     10
aı                           1
ı                            1
i̇                           0
ai̇                          0
Name: count, dtype: int64

In [51]:
not_many_mentions = set(df.entity_lower.value_counts()[lambda x: x < 1000].index)
not_many_mentions.discard("him")
not_many_mentions

{'ai̇', 'artificial ıntelligence', 'aı', 'i̇', 'palm', 'xlnet', 'ı'}

In [52]:
df = df[~(df["entity_lower"].isin(not_many_mentions))].copy()
len(df)

3372808

### Boilerplate detection

Inspect and remove sentences that appear multiple times in the corpus.

In [55]:
with pd.option_context("display.min_rows", 50):
    display(df[df.duplicated(subset="sentence", keep=False)]["sentence"].value_counts()[lambda x: x >= 20])

sentence
I did not use generative AI and AI-assisted technologies in the writing process.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [62]:
boilerplate = df["sentence"].value_counts()[lambda x: x >= 20]
boilerplate_sentences = boilerplate.index
print(f"Boilerplate detection: {len(boilerplate_sentences)} texts appear in corpus at least 20 times, totaling {boilerplate.sum()} rows")

Boilerplate detection: 104 texts appear in corpus at least 20 times, totaling 5283 rows


In [65]:
df = df[~df["sentence"].isin(boilerplate_sentences)].copy()

In [71]:
# How to detect if the same paper has the same text twice?
df[df.duplicated(["id", "match_id", "token_text", "score", "sentence"], keep=False)].sort_values(["id", "match_id"])

,id,year,domain,sent_id,match_id,sentence,masked_sentence_minimal,masked_sentence_chunk,entity,entity_lower,...,token_i,token_text,pos,tag,dep,ent_type,ent_iob,head,entity_type,num_tokens
2213224,W147140176,2007,Physical Sciences,2,0,"If there is a tie, take the result produced by...","If there is a tie, take the result produced by...","If there is a tie, take the result produced by...",system,system,...,78,system,NOUN,NN,pobj,NaN,O,by,artifact,38
2213244,W147140176,2007,Physical Sciences,10,0,"If there is a tie, take the result produced by...","If there is a tie, take the result produced by...","If there is a tie, take the result produced by...",system,system,...,407,system,NOUN,NN,pobj,NaN,O,by,artifact,38
2213225,W147140176,2007,Physical Sciences,2,1,"If there is a tie, take the result produced by...","If there is a tie, take the result produced by...","If there is a tie, take the result produced by...",systems,systems,...,87,systems,NOUN,NNS,poss,NaN,O,order,artifact,38
2213245,W147140176,2007,Physical Sciences,10,1,"If there is a tie, take the result produced by...","If there is a tie, take the result produced by...","If there is a tie, take the result produced by...",systems,systems,...,416,systems,NOUN,NNS,poss,NaN,O,order,artifact,38
2213226,W147140176,2007,Physical Sciences,2,2,"If there is a tie, take the result produced by...","If there is a tie, take the result produced by...","If there is a tie, take the result produced by...",he,he,...,107,he,PRON,PRP,pobj,NaN,O,of,human,38
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3333627,W7138877117,2026,Physical Sciences,30,1,"Within the broader archive, Prototype Prep v0....","Within the broader archive, Prototype Prep v0....","Within the broader archive, Prototype Prep v0....",architecture,architecture,...,952,architecture,NOUN,NN,pobj,NaN,O,from,artifact,29
2202178,W7148730709,2026,Health Sciences,10,0,Results indicate that adult learners are incre...,Results indicate that adult learners are incre...,Results indicate that adult learners are incre...,AI,ai,...,271,AI,PROPN,NNP,compound,ORG,B,use,artifact,22
2202179,W7148730709,2026,Health Sciences,11,0,Most respondents considered generative AI an e...,Most respondents considered generative [MASK] ...,Most respondents considered generative [MASK] ...,AI,ai,...,295,AI,PROPN,NNP,npadvmod,ORG,B,NaN,artifact,25
2202180,W7148730709,2026,Health Sciences,13,0,Results indicate that adult learners are incre...,Results indicate that adult learners are incre...,Results indicate that adult learners are incre...,AI,ai,...,344,AI,PROPN,NNP,compound,ORG,B,use,artifact,22


In [73]:
# Drop the ones that are essentially the same sentence, but for some reason appear in the same document twice
df = df.drop_duplicates(["id", "match_id", "token_text", "score", "sentence"], keep="first").copy()

## Continue with the term `AI`

Back at trying to detect the invalid AI-terms.

In [75]:
len(df[df["entity"] == "AI"])

519286

Due to some earlier inspection, it has been noted that especially the domain Life Sciences is prone to using `AI` in diverging contexts. Some of these instances have already been removed with the earlier step that only includes certain tokens, but the plain `AI` still causes trouble, as well as `AI-`, as anything could be concatenated to it.

In [76]:
print_examples(df[(df["entity"] == "AI") & (df["domain"] == "Life Sciences")])

The arthritis index (AI) was measured and ankle joint structure was analyzed histologically to determine the severity of CIA.
The arthritis index ([MASK]) was measured and ankle joint structure was analyzed histologically to determine the severity of CIA.

Higher soybean equivalent yield was registered under AI-NPOF package (75% RDN through organic sources + two foliar spray of 10% cow urine and vermiwash at 30 and 50 DAS).
Higher soybean equivalent yield was registered under [MASK]-NPOF package (75% RDN through organic sources + two foliar spray of 10% cow urine and vermiwash at 30 and 50 DAS).

For the purpose of determining the most mature fruits or vegetables an AI-based automatic method has been created and developed.
For the purpose of determining the most mature fruits or vegetables an [MASK]-based automatic method has been created and developed.

If we think of AI as sociotechnical systems, then components of values and AI are in the same ontologic category—they are both social

In [77]:
df[df["entity"] == "AI"].groupby("domain", observed=True).size()

domain
Health Sciences      110367
Life Sciences         33030
Physical Sciences    208955
Social Sciences      166928
dtype: int64

In [78]:
df[(df["entity"] == "AI") & (df["domain"] == "Life Sciences")]["pos"].value_counts()

pos
PROPN    32478
ADJ        256
NOUN       211
VERB        71
ADV          7
X            4
ADP          2
INTJ         1
PRON         0
DET          0
PUNCT        0
AUX          0
NUM          0
CCONJ        0
SCONJ        0
SYM          0
PART         0
Name: count, dtype: int64

In [79]:
df[(df["entity"] == "AI") & (df["domain"] == "Life Sciences")]["ent_type"].value_counts()

ent_type
ORG            25932
GPE              384
PERSON           269
EVENT            238
WORK_OF_ART      110
LAW               75
NORP              56
LOC               47
FAC               31
DATE              29
PRODUCT           26
TIME              25
QUANTITY           7
CARDINAL           0
MONEY              0
LANGUAGE           0
PERCENT            0
ORDINAL            0
Name: count, dtype: int64

Trying to inspect if the spaCy annotations could help with detecting the invalid uses of `AI`. But the problem seems to be that there are no clear groups that wouldn't include some incorrectly classified terms.

In [85]:
def print_group_examples(df: pd.DataFrame, group_col):
    print(f"Grouped by column {group_col}\n")
    for name, group in df.groupby(group_col, observed=True):
        print(name, f"({len(group)})")
        sample_size = min(len(group), 10)
        for i, ex in enumerate(group.sample(sample_size, random_state=1000)["sentence"], start=1):
            print(f"{i:>2}) {ex}")
        print()

In [87]:
print_group_examples(df[(df["entity"] == "AI") & (df["domain"] == "Life Sciences")], "ent_type")

Grouped by column ent_type

DATE (29)
 1) Three different models of spray nozzles with air induction (AI 11003, TTI 11003 and AD-IA 11004), three spray pressures (207, 413 and 720 kPa), four different additives for tank mixtures (cane molasses, mineral oil, vegetable oil and glycerin) and the influence of tank mixture stirring time were all evaluated for their effect on EPN (Steinernema feltiae) viability and pathogenicity.
 2) Plasma progesterone concentrations ranged from 3.006.02 ng/ml with an average of 4.42±0.52 ng/ml in pregnant animals and from 0.97-1.91 ng/ml with an average of 1.22±0.06 ng/ml in nonpregnant animals on day 20 post AI.
 3) The ubiquitous use of Shapley values in eXplainable AI (XAI) has been triggered by the tool SHAP, and as a result are commonly referred to as SHAP scores.
 4) Mean time taken for the onset and duration of oestrus were 41.41 1.47 and 14.41 0.43 h; 40.57 2.09 and 18.92 3.69 h for group I and II, respectively and for group III, only two cows show

According to the above inspections, one could exclude at least DATE, EVENT, FAC, LAW, QUANTITY, and TIME. However, this is still likely to leave many invalid terms untouched.

In [88]:
# Does the above hold for other domains as well?
print_group_examples(df[(df["entity"] == "AI") & (df["domain"] != "Life Sciences")], "ent_type")

Grouped by column ent_type

DATE (123)
 1) Notably, image quality significantly improved on AI days (p<0.001).
 2) Going forward, media scholars need to diversify and decolonialise approaches to digital creative labour in the Age of AI by drawing attention to the values and creative practices of global digital cultures.
 3) Methods: This integrative literature review (2020-2025), based on studies from PubMed, Scopus and Web of Science databases, investigated safeguarding Privacy in the Age of AI, focusing on ethical and technical challenges in healthcare data anonymization.
 4) Sonographer mental fatigue was lower despite the increased workload on AI days.
 5) This is just the beginning of AI in medicine.
 6) Ultimately, the article provides valuable insights that reflects an experience of adapting to the teaching and learning Professional English in the epoch of AI.
 7) Notably, the intersection of AI and pulmonary nodule research is concentrated in the following areas: (1) Applicatio

As can be seen above, the validity classification on entity recognition doesn't hold for other domains.

What about POS tags? Can they reveal a structure?

In [89]:
print_group_examples(df[(df["entity"] == "AI")], "pos")

Grouped by column pos

ADJ (2938)
 1) This study develops a novel theoretical framework mapping the nature and purpose of a digitalized AI‐assisted HR ecosystem for delivering exceptional employee experience (EX), an antecedent to employee engagement (EE).
 2) We accomplish this by designing and deploying AI‐driven predictive models that fuse diverse data with siloed domain knowledge.
 3) This study investigated the potential of Artificial Intelligence (AI)-based image recognition to automate and enhance the accuracy, objectivity, and efficiency of CBET practical assessments.
 4) Predicting fading channels is a classical problem with a vast array of applications, including as an enabler of artificial intelligence (AI)-based proactive resource allocation for cellular networks.
 5) Artificial Intelligence (AI)-driven optimization has emerged as a transformative solution for improving water usage and waste management by leveraging real-time data analytics, predictive modeling, and automat

In [90]:
print_group_examples(df[(df["entity"] == "AI")], "dep")

Grouped by column dep

ROOT (512)
 1) Generative AI as a silver bullet.
 2) Designing Tomorrow: AI, Ethics, and Society's Next Steps, 125.
 3) AI in teaching geography, as well as its potential for enhancing learning strategies.
 4) AI & ML of a chatbot store a products detail and an assistant interact the PC with farmer in their natural language.
 5) VRP termasuk kedalam AI (Artificial Intelligence).
 6) AI, IoT, and satellite data drive smarter, resilient agriculture.
 7) AI (rapid processing),synchronized through artifacts; mechanism stability confirmed by substantial empiricalfoundation.
 8) Maas AI, Stocchetti N, Bullock R (August 2008).
 9) Casas AI, Hassan AA, Larsen SJ, Gomez-Rangel V, Elbatreek M, Kleikers PWM, Guney E, Egea J, López MG, Baumbach J, Schmidt HHHW.
10) RO1 AI 52313).Competing

acl (5)
 1) While the challenge of distinguishing AI‐generated from real images is widely acknowledged, the specific cognitive biases that systematically shape human judgment in this domai

So far, excluding by entity recognition in the Life Sciences domain seems the most appropriate, but one could also try simple pattern-matching, as a big portion of the invalid terms seem to be related to "artificial insemination" in cows.

In [92]:
exclude = [r"\bcows?\b",
           r"\bsemen\b",
           r"\bsperm\b",
           r"\binsemin",
           r"\bfertil",
           r"\bmating\b",
           r"\bfarm\b",
           r"\bbred\b",
           r"\bbreed\b",
           r"\bdairy\b",
           r"\bcattle\b",
           r"\bbulls?\b",
           r"\bboars?\b",
           r"\bpregnan",
           r"\bP/AI\b",
           r"\bAI-1\b",
           r"\bAI-2\b",
           r"\bacre\b",
           r"\bfl oz\b",
           r"\bmg\b",
           r"\bactivity index\b"
        ]
pattern = "|".join(exclude)
cows = df[(df["entity"] == "AI") & (df["sentence"].str.contains(pattern, case=False))]
print(len(cows))

6296


In [94]:
for i, sent in enumerate(cows.sample(100)["sentence"], start=1):
    print(f"{i:>2}) {sent}")

 1) Comparing overall intraspecific and interspecific AI, pregnancy rates were 25/37 (67.6%) and 14/63 (22.2%), respectively (p = 0.0001).
 2) Our objective was to compare the expression of target genes in the endometrium of 15-d pregnant cows that expressed estrus at timed-AI (EST) with those that did not express estrus (NOEST).
 3) We analyzed publicly available data to investigate relationships between agriculture (the number one land use purpose in the United States), Native and Western food securities, and the occurrence of AI/AN pregnancy-related maternal mortality.
 4) Experimental groups of fish were prescribed with 'Rybozuril' in a dose (by AI) of 50 and 10 mg/kg for two consecutive days.
 5) Thirty-six hours after the second PGF2α injection, goats were either naturally bred (n = 14) with the same breed mature buck or artificially inseminated (n = 15) with frozen/thawed semen using cervical AI.
 6) To improve our artificial insemination (AI) success, the distribution of sperm 

In [97]:
for i, sent in enumerate(cows[cows["domain"] != "Life Sciences"].sample(20)["sentence"], start=1):
    print(f"{i:>2}) {sent}")

 1) This study presents Farm and Learn, an offline-first mobile learning system that integrates Augmented Reality (AR), Artificial Intelligence (AI), and Game-Based Learning (GBL) to enhance agricultural education among children in low-connectivity environments.
 2) The eCG treatment (300 or 400 IU) administered at the time of P4 removal increased P/AI in anestrus suckled Bos taurus beef cows.
 3) Herein, we present an AI-driven workflow for discovering high-performance Mg cathode materials.
 4) AI will augment knowledge and assist nurses for decision making in various areas of obstetrics and gynecological field such as fetal heart monitoring and pregnancy surveillance, predicting preterm labor, in vitro fertilization, gynecological surgery, screening of cervical and ovarian cancers and screening for gestational diabetes mellitus.
 5) Hence, it is possible to increase the efficiency of goat breeding by manipulating the seasonal variations of semen quality for immediate AI and/or cryopr

The keyword-based approach will result also in false positives, but I will be using it.

In [99]:
cows.domain.value_counts()

domain
Life Sciences        4522
Health Sciences      1261
Physical Sciences     333
Social Sciences       180
Name: count, dtype: int64

In [102]:
cows.token_text.value_counts()

token_text
AI                 6278
AI)-based            10
AI-                   3
intelligence(AI       2
AI)-driven            2
AI‐based              1
Name: count, dtype: int64

In [103]:
cows = cows[cows["token_text"] == "AI"].copy()
len(cows)

6278

In [105]:
 df = df.drop(cows.index).copy()

See if the entity type -based drop should also be performed

In [107]:
print_group_examples(df[(df["entity"] == "AI") & (df["domain"] == "Life Sciences")], "ent_type")

Grouped by column ent_type

DATE (15)
 1) Recently, it was reported that aryl 2-aminoimidazole (2-AI) compounds potentiate macrolide antibiotics against a highly virulent strain of A. baumannii, AB5075.
 2) Three different models of spray nozzles with air induction (AI 11003, TTI 11003 and AD-IA 11004), three spray pressures (207, 413 and 720 kPa), four different additives for tank mixtures (cane molasses, mineral oil, vegetable oil and glycerin) and the influence of tank mixture stirring time were all evaluated for their effect on EPN (Steinernema feltiae) viability and pathogenicity.
 3) Follicle sizes were taken using ultrasound on Days 0, 7, 9 and 10 and estrous signs were taken on the Day of AI.
 4) Supported by NIH grants AI 079705, AI 105813/105813S1, AI138944, AI 167416 and Lupus Research Alliance grant LRA 641363 to PC.
 5) [1.12-1.79]), AI (1.31 [1.05-1.65]), and 90 d post AI (1.38 [1.10-1.74]).
 6) This can contribute to ongoing developments in neuro-AI and self-organized mu

In [108]:
for sent in (df[(df["entity"] == "AI") & (df["domain"] == "Life Sciences") & (df["ent_type"].isin(["DATE", "EVENT", "FAC", "LAW", "QUANTITY", "TIME"]))]
    .sample(50)["sentence"]
):
    print("*", sent)

* This editorial process and these questions also contributed to the "AI Provocateurs" panel for the a2ru community in early October 2023, and informed plans for the "Generative AI and Speculative Futures for the Arts: Pushback and Possibilities" panel at the annual a2ru conference.
* Logistic regression analysis likewise revealed that Select AI protocol, pre-ovulatory follicle size on the day of AI, and estrous signs are significant factors on the success of FTAI program in water buffaloes.
* Carbaryl SOS applied at 2 lb AI and Parathion 8EC applied at 9 oz AI were the standards.
* If an increase in P4 levels was to be observed, a second larger trial was to be conducted testing the effect of post-AI GnRH on CR at return to estrus.
* Multivariable logistic models were offered with the following predictors: year of LEL (2011 through 2015), season of LEL (summer vs. fall vs. winter vs. spring), parity (1 vs. 2 vs. ≥3), uterine disease (UD), non-uterine disease (NUD), body condition score

In [109]:
len(df[(df["entity"] == "AI") & (df["domain"] == "Life Sciences") & (df["ent_type"].isin(["DATE", "EVENT", "FAC", "LAW", "QUANTITY", "TIME"]))])

214

In [112]:
remove_in_life_sciences = set(["DATE", "EVENT", "FAC", "LAW", "QUANTITY", "TIME"])
df = df[~((df["entity"] == "AI") & (df["domain"] == "Life Sciences") & (df["ent_type"].isin(remove_in_life_sciences)))].copy()

In [113]:
for entity in df["entity_lower"].unique().tolist():
    print(df[df["entity_lower"] == entity].token_text.value_counts()[lambda x: x >= 10])
    print()

token_text
agent          30300
Agent           3092
multi‐agent      556
agent(s           14
agent‐based       12
Name: count, dtype: int64

token_text
systems      131386
Systems        5610
systems.1        11
systems-         10
Name: count, dtype: int64

token_text
model          183047
Model            7597
model-             43
model‐based        24
model(s            21
model‐free         12
Name: count, dtype: int64

token_text
approach    88417
Approach      604
Name: count, dtype: int64

token_text
models    150902
Models      2304
Name: count, dtype: int64

token_text
intelligence           140643
artificial             128449
Artificial              48809
Intelligence            36231
intelligence(AI            90
Intelligence(AI            73
intelligence‐based         66
intelligence-              26
AI(Artificial              17
intelligence‐driven        14
AI(artificial              11
intelligence‐              10
Name: count, dtype: int64

token_text
chatbot    119

Have a look at a random sample; if everything looks good here, I am moving forward.

In [114]:
print_examples(df, 50)

The emergence of large language models (LLMs) has spurred numerous applications across various domains, including material design.
The emergence of [MASK] (LLMs) has spurred numerous applications across various domains, including material design.

Recent progress in visual image–language models has marked significant advancements in multimodal reasoning tasks.
Recent progress in visual image–[MASK] has marked significant advancements in multimodal reasoning tasks.

Incorporating AI into healthcare systems has the ability not only to improve diagnosis and treatment procedures, but also to optimize resource allocation.
Incorporating AI into healthcare [MASK] has the ability not only to improve diagnosis and treatment procedures, but also to optimize resource allocation.

DTs further advance these functions through bi-directional connectivity and higher-fidelity physical models, supporting adaptive control, predictive maintenance, and system-level optimization.
DTs further advance these f

In [119]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3359491 entries, 0 to 3374907
Data columns (total 25 columns):
 #   Column                   Dtype   
---  ------                   -----   
 0   id                       object  
 1   year                     int64   
 2   domain                   category
 3   sent_id                  int64   
 4   match_id                 int64   
 5   sentence                 object  
 6   masked_sentence_minimal  object  
 7   masked_sentence_chunk    object  
 8   entity                   object  
 9   entity_lower             category
 10  in_noun_chunk            bool    
 11  chunk_text               object  
 12  chunk_root_text          object  
 13  chunk_root_dep           category
 14  score                    float64 
 15  token_i                  int64   
 16  token_text               object  
 17  pos                      category
 18  tag                      category
 19  dep                      category
 20  ent_type                 cate

In [118]:
df.to_csv("new-method-cleaned.csv", index=False)

In [2]:
cat_cols = "domain,entity_lower,chunk_root_dep,pos,tag,dep,ent_type,ent_iob,entity_type".split(",")
dtypes = {col: "category" for col in cat_cols}
df = pd.read_csv("new-method-cleaned.csv", dtype=dtypes)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3359491 entries, 0 to 3359490
Data columns (total 25 columns):
 #   Column                   Dtype   
---  ------                   -----   
 0   id                       object  
 1   year                     int64   
 2   domain                   category
 3   sent_id                  int64   
 4   match_id                 int64   
 5   sentence                 object  
 6   masked_sentence_minimal  object  
 7   masked_sentence_chunk    object  
 8   entity                   object  
 9   entity_lower             category
 10  in_noun_chunk            bool    
 11  chunk_text               object  
 12  chunk_root_text          object  
 13  chunk_root_dep           category
 14  score                    float64 
 15  token_i                  int64   
 16  token_text               object  
 17  pos                      category
 18  tag                      category
 19  dep                      category
 20  ent_type                

## Proper noun LLM check

As the final post-processing step, remove instances of the proper noun LLMs that appear before the year each model was introduced.

In [23]:
df[df["entity_type"] == "lm"]["entity_lower"].unique().tolist()

['chatbot',
 'language model',
 'lm',
 'language models',
 'lms',
 'chatbots',
 'large language models',
 'claude',
 'transformers',
 'transformer',
 'pre-trained language model',
 'pre-trained language models',
 'llms',
 'bert',
 'gpt',
 'large language model',
 'genai',
 'foundation models',
 'foundation model',
 'llm',
 'gemini',
 'chatgpt',
 'llama']

In [26]:
print("Earliest appearance per term")
for entity in df[df["entity_type"] == "lm"]["entity_lower"].unique().tolist() + ["artificial intelligence", "ai"]:
    print(f"{entity} -> {df[df["entity_lower"] == entity]["year"].min()}")

Earliest appearance per term
chatbot -> 2003
language model -> 1973
lm -> 1872
language models -> 1987
lms -> 1911
chatbots -> 1996
large language models -> 2008
claude -> 2006
transformers -> 1999
transformer -> 1998
pre-trained language model -> 1862
pre-trained language models -> 2016
llms -> 2012
bert -> 2018
gpt -> 1986
large language model -> 2012
genai -> 2018
foundation models -> 2020
foundation model -> 2021
llm -> 2008
gemini -> 1989
chatgpt -> 2001
llama -> 2018
artificial intelligence -> 1907
ai -> 1879


In [28]:
def get_term_stats(df: pd.DataFrame, term: str, year: int):
    mask = (df["entity_lower"] == term) & (df["year"] < year)
    subset = df[mask]
    invalid = len(subset)
    print(f"{term} instances before {year}: {invalid}")
    if invalid > 0:
        example = subset.sample(1)
        print("Example:")
        print(example["sentence"].item(), example["year"].item())
    return df[~mask].copy()

term_year_pairs = (
    ("artificial intelligence", 1955),
    ("ai", 1955),
    ("transformer", 2017),
    ("transformers", 2017),
    ("pre-trained language model", 2018),
    ("pre-trained language models", 2018),
    ("bert", 2018),
    ("gpt", 2018),
    ("foundation model", 2021),
    ("foundation models", 2021),
    ("chatgpt", 2022),
    ("llama", 2023),
    ("claude", 2023),
    ("gemini", 2023)
)

print("DataFrame size before:", len(df))
for term, year in term_year_pairs:
    df = get_term_stats(df, term, year)
print("DataFrame size after:", len(df))

DataFrame size before: 3359491
artificial intelligence instances before 1955: 2
Example:
Herein, we propose MIXTURE (huMan-In-the-loop eXplainable artificial intelligence Through the Use of REcurrent training), a method to develop deep learning models for extracting pathologically significant findings based on an expert pathologist's perspective with a small annotation effort. 1907
ai instances before 1955: 15
Example:
Industrial Pure AI Sheets(1) 1953
transformer instances before 2017: 46
Example:
In transformer fault diagnosis, dissolved gas analysis (DGA) is been widely employed for a long period and numerous methods have been innovated to interpret its results. 2014
transformers instances before 2017: 28
Example:
Abstract Transformers are expensive and important equipment in the whole power system. 2015
pre-trained language model instances before 2018: 5
Example:
IgFold consists of a pre-trained language model trained on 558M natural antibody sequences followed by graph networks th

In [29]:
df.to_csv("new-method-cleaned.csv", index=False)